In [4]:
import torch
import pandas as pd
import pickle
import torch.nn as nn
from pykeen.models import ERModel
from pykeen.triples import TriplesFactory
from pykeen.nn.modules import DistMultInteraction
from sklearn.decomposition import PCA
from pykeen.losses import MarginRankingLoss
from pykeen.evaluation import RankBasedEvaluator
from pykeen.training import SLCWATrainingLoop
from pykeen.nn.representation import Representation

In [5]:
class MLPRepresentation(Representation):
    def __init__(self, *, max_id: int, base: nn.Embedding, hidden_dim: int, dropout: float = 0.2):
        dim = base.embedding_dim
        super().__init__(max_id=max_id, shape=(dim,))
        self.base = base
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
        )

    def _plain_forward(self, indices: torch.LongTensor | None) -> torch.FloatTensor:
        if indices is None:
            x = self.base.weight  # [max_id, dim]
        else:
            x = self.base(indices)  # [*, dim]
        return self.mlp(x)

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
main_data = pd.read_csv('data/PPI_1M.csv')
main_data = main_data.astype(str)

triples = main_data[['protein_1', 'relation', 'protein_2']].values
triplet_data = TriplesFactory.from_labeled_triples(triples, create_inverse_triples=True)
training_set, testing_set, validation_set = triplet_data.split([0.8, 0.1, 0.1], random_state=100)

In [8]:
with open("data/dict.pkl", "rb") as f:
    emb_dict = pickle.load(f)

In [13]:
entity2id = list(triplet_data.entity_to_id.keys())
EMB = torch.stack([emb_dict[int(raw_id)] for raw_id in entity2id], dim=0)

pca = PCA(n_components=64)        # число компонент
X_pca = pca.fit_transform(EMB.numpy())   # X: torch.Tensor -> numpy
EMB = torch.from_numpy(X_pca)

In [14]:
D = EMB.shape[1]

entity_emb = nn.Embedding(triplet_data.num_entities, D)
with torch.no_grad():
    entity_emb.weight.copy_(EMB)
entity_emb.weight.requires_grad_(False)

ent = MLPRepresentation(
    max_id=triplet_data.num_entities,
    base=entity_emb,
    hidden_dim=128,
    dropout=0.2,
)

rel_emb = nn.Embedding(triplet_data.num_relations, D)
rel = MLPRepresentation(
    max_id=triplet_data.num_relations,
    base=rel_emb,
    hidden_dim=128,
    dropout=0.2,
)



In [15]:
EMB_DIM = EMB.shape[1]
LR = 1e-3
MARGIN = 1.1
WEIGHT_DECAY = 1e-3
EPOCHS = 10
BATCH_SIZE = 4096
NUM_NEGS_PER_POS = 10

loss_function = MarginRankingLoss(margin=MARGIN)

model = ERModel(
    triples_factory=triplet_data,
    interaction=DistMultInteraction(),
    entity_representations=ent,
    relation_representations=rel,
    loss=loss_function,
    random_seed=100
)
model = model.to(device)

#2) заморозить entity-эмбеддинги
params = []
params += list(ent.parameters())
params += list(rel.parameters())

optimizer = torch.optim.Adam(params, lr=1e-3, weight_decay=WEIGHT_DECAY)

training_loop = SLCWATrainingLoop(
    model=model,
    triples_factory=training_set,
    optimizer=optimizer,
    negative_sampler='pseudotyped',
    negative_sampler_kwargs=dict(
        num_negs_per_pos=NUM_NEGS_PER_POS
    )
)

training_loop.train(
    num_epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    triples_factory=training_set,
    use_tqdm_batch=False,
)

evaluator = RankBasedEvaluator()
#
model_results = evaluator.evaluate(
    model=model,
    mapped_triples=testing_set.mapped_triples.to(device),
    additional_filter_triples=[
            training_set.mapped_triples.to(device),
            validation_set.mapped_triples.to(device),
        ],

)

metrics = model_results.to_df()
metrics = metrics[(metrics['Side'] == 'both') & (metrics['Rank_type'] == 'realistic')]
metrics

Training epochs on cuda:0: 100%|██████████| 10/10 [01:06<00:00,  6.65s/epoch, loss=0.823, prev_loss=0.836]
Evaluating on cuda:0: 100%|██████████| 100k/100k [00:42<00:00, 2.37ktriple/s] 


,Side,Rank_type,Metric,Value
5,both,realistic,median_rank,4.536000e+03
14,both,realistic,z_geometric_mean_rank,3.548205e+02
23,both,realistic,inverse_geometric_mean_rank,3.516736e-04
32,both,realistic,z_inverse_harmonic_mean_rank,4.671195e+02
41,both,realistic,inverse_harmonic_mean_rank,7.228869e-03
50,both,realistic,count,1.999960e+05
59,both,realistic,adjusted_geometric_mean_rank_index,7.927977e-01
68,both,realistic,adjusted_inverse_harmonic_mean_rank,6.933116e-03
77,both,realistic,harmonic_mean_rank,1.383342e+02
86,both,realistic,arithmetic_mean_rank,8.820676e+03
